In [1]:
import pandas as pd
import os

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [8]:
gear_df = pd.read_parquet(r'Deduplicate/merged_all_deduplicated.parquet')


def _fix_mojibake(value):
    if not isinstance(value, str):
        return value
    if not any(marker in value for marker in ('Ã', 'Ð', 'Ñ', 'â')):
        return value
    repaired_bytes = bytearray()
    for character in value:
        code_point = ord(character)
        if code_point <= 255:
            repaired_bytes.append(code_point)
            continue
        try:
            repaired_bytes.extend(character.encode('cp1252'))
        except UnicodeEncodeError:
            return value
    try:
        return bytes(repaired_bytes).decode('utf-8')
    except UnicodeError:
        return value


unique_gear = (
    gear_df['gear']
    .dropna()
    .map(_fix_mojibake)
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

output_path = r'Gear/gear_types.xlsx'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

pd.DataFrame({'gear': unique_gear}).to_excel(output_path, index=False)

In [ ]:
# filter out records whose gear matches any gear marked 'x' in the manual Excel file
manual_path = r'Gear/gear_types_manual.xlsx'
save_path = r'Gear/merged_all_gear.parquet'

manual = pd.read_excel(manual_path)

# normalize gear strings using existing _fix_mojibake
manual_gear_norm = manual['gear'].map(_fix_mojibake).fillna('').astype(str).str.strip()
remove_mask = manual['remove'].astype(str).str.lower().eq('x')
remove_set = set(manual_gear_norm[remove_mask])

initial_count = len(gear_df)

gear_norm = gear_df['gear'].map(_fix_mojibake).fillna('').astype(str).str.strip()
filtered_df = gear_df[~gear_norm.isin(remove_set)].copy()

after_count = len(filtered_df)

os.makedirs(os.path.dirname(save_path), exist_ok=True)
filtered_df.to_parquet(save_path, index=False)

print(f'before: {initial_count}')
print(f'after:  {after_count}')